# Sistema Inteligente — Colônia Aurora Siger

## Notebook PRINCIPAL (Capítulo 1 — Desafio Integrado)

Este notebook é **autossuficiente**: contém o código de todos os
módulos do projeto integrado em uma única execução.

**Capítulos aplicados (toda a fase):**
- **Cap. 2** — Lógica booleana, mintermos, simplificação
- **Cap. 3** — Subalgoritmos, parâmetros default, recursividade
- **Cap. 4** — Tabela hash, hierarquia/árvore
- **Cap. 5** — Vetores 1D como listas Python
- **Cap. 6** — Ciclo BUSCAR → DECODIFICAR → EXECUTAR
- **Cap. 7** — Regressão linear (mínimos quadrados, R²)
- **Cap. 8** — Potência eólica ($P = \\tfrac12 \\rho A v^3$),
  Betz, curva cut-in/cut-out, geração solar

Tudo em Python puro, sem bibliotecas externas.

## Parte 1 — Organização dos dados (caps. 4 e 5)

In [ ]:
def criar_colonia():
    return {
        "energetico": {
            "solar": {
                "potencia_atual": 45,
                "historico_irradiancia": [120, 180, 240, 300, 360, 420, 480],
                "historico_geracao":     [12,  19,  26,  33,  40,  47,  54],
                "area_paineis_m2": 20, "eficiencia": 0.18,
            },
            "eolico": {
                "potencia_atual": 25,
                "historico_vento":   [8, 10, 12, 13, 15],
                "historico_geracao": [18, 21, 25, 27, 31],
                "area_rotor_m2": 12, "rho_ar": 0.020,
                "cut_in": 3, "v_nominal": 12, "cut_out": 25, "cp": 0.40,
            },
            "reserva_baterias": 60, "capacidade_baterias": 100,
        },
        "ambiental": {
            "temperatura_interna": 22, "temperatura_externa": -55,
            "velocidade_vento": 14, "irradiancia_atual": 360,
            "previsao_tempestade": False,
        },
        "operacional": {
            "consumo_total": 70,
            "modulos": {
                "suporte_vida": {"consumo": 30, "essencial": True},
                "comunicacao":  {"consumo": 8,  "essencial": True},
                "estufa":       {"consumo": 20, "essencial": False},
                "laboratorio":  {"consumo": 12, "essencial": False},
            },
        },
        "sensores": {
            "S001": {"tipo": "temperatura", "valor": 22,  "ok": True},
            "S002": {"tipo": "pressao",     "valor": 101, "ok": True},
            "S003": {"tipo": "oxigenio",    "valor": 21,  "ok": True},
            "S004": {"tipo": "umidade",     "valor": 45,  "ok": True},
        },
    }


def acessar(colonia, caminho):
    atual = colonia
    for chave in caminho:
        if isinstance(atual, dict) and chave in atual:
            atual = atual[chave]
        else:
            return None
    return atual


def funcao_hash(chave, tamanho):
    if isinstance(chave, int):
        return chave % tamanho
    return sum(ord(c) for c in str(chave)) % tamanho


def buscar_sensor(colonia, id_sensor):
    return colonia["sensores"].get(id_sensor)


def energia_total_gerada(colonia):
    return (acessar(colonia, ["energetico", "solar", "potencia_atual"]) +
            acessar(colonia, ["energetico", "eolico", "potencia_atual"]))

## Parte 2 — Lógica booleana (cap. 2)

In [ ]:
def AND(a, b): return 1 if (a == 1 and b == 1) else 0
def OR(a, b):  return 1 if (a == 1 or b == 1)  else 0
def NOT(a):    return 1 - a


def avaliar_condicoes(energia, consumo, previsao_tempestade):
    return {
        "energia_critica": 1 if energia < 30 else 0,
        "energia_baixa":   1 if energia < 50 else 0,
        "consumo_alto":    1 if consumo >= 60 else 0,
        "tempestade":      1 if previsao_tempestade else 0,
    }


def expressao_modo_emergencia(cond):
    return AND(cond["energia_critica"], cond["consumo_alto"])


def expressao_modo_economia(cond):
    p1 = AND(cond["tempestade"], cond["energia_baixa"])
    p2 = AND(AND(cond["energia_baixa"], cond["consumo_alto"]),
             NOT(cond["energia_critica"]))
    return OR(p1, p2)

## Parte 3 — Regressão linear (cap. 7)

$\\beta_1 = \\frac{\\sum (x_i - \\bar{x})(y_i - \\bar{y})}{\\sum (x_i - \\bar{x})^2}$,
$\\beta_0 = \\bar{y} - \\beta_1 \\bar{x}$,
$R^2 = 1 - \\frac{\\sum (y_i - \\hat{y}_i)^2}{\\sum (y_i - \\bar{y})^2}$

In [ ]:
def ajustar_reta(x, y):
    if len(x) != len(y) or len(x) == 0:
        raise ValueError("Listas invalidas.")
    x_med = sum(x) / len(x)
    y_med = sum(y) / len(y)
    num = sum((x[i] - x_med) * (y[i] - y_med) for i in range(len(x)))
    den = sum((xi - x_med) ** 2 for xi in x)
    if den == 0:
        return y_med, 0.0
    b1 = num / den
    b0 = y_med - b1 * x_med
    return b0, b1


def prever(b0, b1, x_novo):
    return b0 + b1 * x_novo


def r_quadrado(x, y, b0, b1):
    y_med = sum(y) / len(y)
    sqe = sum((y[i] - (b0 + b1 * x[i])) ** 2 for i in range(len(x)))
    sqt = sum((yi - y_med) ** 2 for yi in y)
    return 1.0 if sqt == 0 else 1 - sqe / sqt


def prever_energia_eolica(historico_vento, historico_geracao, vento_previsto):
    b0, b1 = ajustar_reta(historico_vento, historico_geracao)
    return {
        "vento_previsto": vento_previsto,
        "energia_estimada": round(prever(b0, b1, vento_previsto), 2),
        "beta0": round(b0, 3), "beta1": round(b1, 3),
        "r2": round(r_quadrado(historico_vento, historico_geracao, b0, b1), 3),
    }

## Parte 4 — Energia renovável (cap. 8)

$P = \\tfrac{1}{2} \\rho A v^3$, com Cp ≤ Betz = 16/27 ≈ 0.593.

In [ ]:
BETZ = 16 / 27


def potencia_eolica(velocidade, rho, area, cp,
                    cut_in, v_nominal, cut_out):
    if cp > BETZ:
        cp = BETZ
    if velocidade < cut_in or velocidade >= cut_out:
        return 0.0
    p = 0.5 * rho * area * (velocidade ** 3) * cp
    p_nominal = 0.5 * rho * area * (v_nominal ** 3) * cp
    return p_nominal if p > p_nominal else p


def classificar_operacao_turbina(v, cut_in, v_nominal, cut_out):
    if v < cut_in:        return "PARADA (vento fraco)"
    elif v < v_nominal:   return "GERANDO (regiao cubica)"
    elif v < cut_out:     return "NOMINAL (saturada)"
    else:                 return "DESLIGADA (vento forte)"


def potencia_solar(irradiancia, area, eficiencia):
    if irradiancia <= 0 or area <= 0 or eficiencia <= 0:
        return 0.0
    return irradiancia * area * eficiencia

## Parte 5 — Decisões (caps. 2 e 3)

In [ ]:
def analisar_energia(geracao, consumo, reserva=0):
    saldo = geracao - consumo
    if consumo > geracao:
        s, m = "RISCO", "ALERTA: consumo maior que geracao"
    elif geracao > consumo:
        s, m = "SOBRA", "SUGESTAO: armazenar energia excedente"
    else:
        s, m = "EQUILIBRIO", "OK: geracao e consumo equilibrados"
    return {"geracao": geracao, "consumo": consumo, "reserva": reserva,
            "saldo": saldo, "situacao": s, "mensagem": m}


def decidir_acao(energia, consumo, previsao_tempestade=False):
    consumo_alto    = consumo >= 60
    energia_critica = energia < 30
    energia_baixa   = energia < 50
    if energia_critica and consumo_alto:
        return {"nivel": "CRITICO", "acao": "ATIVAR MODO DE EMERGENCIA",
                "detalhe": "Energia critica com consumo alto."}
    elif previsao_tempestade and energia_baixa:
        return {"nivel": "ALTO", "acao": "ATIVAR MODO DE ECONOMIA",
                "detalhe": "Tempestade prevista e energia baixa."}
    elif energia_baixa:
        return {"nivel": "MEDIO", "acao": "REDUZIR CONSUMO",
                "detalhe": "Energia abaixo do nivel seguro (50)."}
    else:
        return {"nivel": "NORMAL", "acao": "MANTER SISTEMAS NORMAIS",
                "detalhe": "Condicoes dentro do esperado."}


def priorizar_modulos(modulos, modo_economia):
    ligados, desligados, consumo_final = [], [], 0
    for nome in modulos:
        info = modulos[nome]
        if info["essencial"]:
            ligados.append(nome); consumo_final += info["consumo"]
        elif modo_economia:
            desligados.append(nome)
        else:
            ligados.append(nome); consumo_final += info["consumo"]
    return {"ligados": ligados, "desligados": desligados,
            "consumo_final": consumo_final}


def propagar_alerta(setores, indice=0, mensagem="ALERTA"):
    if indice >= len(setores):
        return []
    return ([f"[{mensagem}] -> setor '{setores[indice]}'"] +
            propagar_alerta(setores, indice + 1, mensagem))

## Parte 6 — Ciclo BUSCAR → DECODIFICAR → EXECUTAR (cap. 6)

Inspirado no ciclo clássico de von Neumann:
1. **BUSCAR** = ler sensores (entrada)
2. **DECODIFICAR** = analisar logicamente (processamento)
3. **EXECUTAR** = aplicar ação (saída)

In [ ]:
def buscar(colonia):
    return {
        "geracao": energia_total_gerada(colonia),
        "consumo": acessar(colonia, ["operacional", "consumo_total"]),
        "reserva": acessar(colonia, ["energetico", "reserva_baterias"]),
        "tempestade": acessar(colonia, ["ambiental", "previsao_tempestade"]),
        "vento": acessar(colonia, ["ambiental", "velocidade_vento"]),
        "irradiancia": acessar(colonia, ["ambiental", "irradiancia_atual"]),
        "modulos": acessar(colonia, ["operacional", "modulos"]),
    }


def decodificar(leituras, colonia):
    analise = analisar_energia(leituras["geracao"], leituras["consumo"],
                                leituras["reserva"])
    cond = avaliar_condicoes(leituras["geracao"], leituras["consumo"],
                              leituras["tempestade"])
    emergencia = expressao_modo_emergencia(cond)
    economia = expressao_modo_economia(cond)
    hist_v = acessar(colonia, ["energetico", "eolico", "historico_vento"])
    hist_g = acessar(colonia, ["energetico", "eolico", "historico_geracao"])
    prev = prever_energia_eolica(hist_v, hist_g, leituras["vento"])
    e = acessar(colonia, ["energetico", "eolico"])
    p_eolica = potencia_eolica(leituras["vento"], e["rho_ar"],
                                e["area_rotor_m2"], e["cp"],
                                e["cut_in"], e["v_nominal"], e["cut_out"])
    estado_t = classificar_operacao_turbina(leituras["vento"], e["cut_in"],
                                             e["v_nominal"], e["cut_out"])
    s = acessar(colonia, ["energetico", "solar"])
    p_solar = potencia_solar(leituras["irradiancia"], s["area_paineis_m2"],
                              s["eficiencia"])
    return {"analise": analise, "cond_booleanas": cond,
            "emergencia": emergencia, "economia": economia,
            "previsao": prev, "pot_eolica_fisica": p_eolica,
            "estado_turbina": estado_t, "pot_solar_fisica": p_solar}


def executar(leituras, info):
    decisao = decidir_acao(leituras["geracao"], leituras["consumo"],
                            leituras["tempestade"])
    modo_economia = decisao["nivel"] in ("CRITICO", "ALTO", "MEDIO")
    prioridade = priorizar_modulos(leituras["modulos"], modo_economia)
    return {"decisao": decisao, "modo_economia": modo_economia,
            "prioridade": prioridade}

## Parte 7 — Execução do sistema

In [ ]:
colonia = criar_colonia()

leituras = buscar(colonia)
print("[BUSCAR] leituras dos sensores:")
for k, v in leituras.items():
    if k != "modulos":
        print(f"    {k:14s} = {v}")

info = decodificar(leituras, colonia)
print()
print("[DECODIFICAR] processamento:")
print(f"    {info['analise']['mensagem']}")
print(f"    Booleanas: {info['cond_booleanas']}")
print(f"    Modo emergencia (AND) = {info['emergencia']}")
print(f"    Modo economia (OR/AND) = {info['economia']}")
prev = info["previsao"]
print(f"    Reta: energia = {prev['beta1']} * vento + {prev['beta0']}  "
      f"(R2 = {prev['r2']})")
print(f"    Previsao p/ vento={leituras['vento']}: ~{prev['energia_estimada']}")
print(f"    Potencia eolica fisica: {info['pot_eolica_fisica']:.2f} W  "
      f"[{info['estado_turbina']}]")
print(f"    Potencia solar fisica:  {info['pot_solar_fisica']:.2f} W")

saida = executar(leituras, info)
print()
print("[EXECUTAR] acao:")
print(f"    Nivel: {saida['decisao']['nivel']}")
print(f"    Acao : {saida['decisao']['acao']}")
print(f"    Modulos LIGADOS .: {saida['prioridade']['ligados']}")
print(f"    Modulos DESLIGADOS: {saida['prioridade']['desligados']}")

## Parte 8 — Demonstração de cenários

Aciona os quatro níveis de decisão da lógica booleana combinada.

In [ ]:
cenarios = [
    ("Energia critica + consumo alto",       25, 70, False),
    ("Tempestade prevista + energia baixa",  45, 40, True),
    ("Energia baixa, sem tempestade",        48, 35, False),
    ("Condicoes normais",                    90, 50, False),
]
for nome, e, c, t in cenarios:
    d = decidir_acao(e, c, t)
    print(f"- {nome}")
    print(f"  Entrada: energia={e}, consumo={c}, tempestade={t}")
    print(f"  Saida  : [{d['nivel']}] {d['acao']}")
    print()

## Parte 9 — Curva eólica em Marte (cap. 8)

Demonstra a curva cut-in / nominal / cut-out com a densidade do ar
marciana (~0.020 kg/m³).

In [ ]:
print("Velocidade | Estado                    | Potencia (W)")
print("-----------+---------------------------+-------------")
for v in [0, 2, 5, 10, 12, 15, 20, 26]:
    estado = classificar_operacao_turbina(v, 3, 12, 25)
    p = potencia_eolica(v, 0.020, 12, 0.40, 3, 12, 25)
    print(f"  {v:>5} m/s | {estado:25s} | {p:>10.3f}")

## Parte 10 — Recursividade e tabela hash (caps. 3 e 4)

In [ ]:
print("Propagacao recursiva de alertas (cap. 3):")
for a in propagar_alerta(["habitat-1", "estufa", "lab", "energia"],
                          mensagem="ALERTA O2 BAIXO"):
    print("  " + a)

print()
print("Acesso O(1) por hash (cap. 4):")
for sid in ["S001", "S003", "S999"]:
    s = buscar_sensor(colonia, sid)
    print(f"  Sensor {sid}: {s if s else 'NAO ENCONTRADO'}")
print(f"  funcao_hash('S001', 7) = {funcao_hash('S001', 7)}")